In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel, PatchOp
from pyspark.sql import Row
from pyspark.sql.functions import col, collect_set, array_contains

In [0]:
_ambiente = "PRD"

if 'prd' in _ambiente.lower():
    ambiente = _ambiente.lower()
elif 'qa' in _ambiente.lower():
    ambiente = _ambiente.lower()
else:
    ambiente = 'dev'

print("AMBIENTE: ",ambiente)

In [0]:
# name of workflows that will not receive permission to manage
dbutils.widgets.text("ListJobsExclude", "pipe_desativa_workflows")

## Get permission from SDK

In [0]:
secret_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-secret")
tenant_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-tenant")
clientId_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-application")
host_dtb = dbutils.secrets.get(scope=f"az-keyvault", key=f"adb-host")

w = WorkspaceClient(
    host = host_dtb,
    azure_tenant_id = tenant_dtb,
    azure_client_id = clientId_dtb,
    azure_client_secret = secret_dtb)

In [0]:
# performs permission update in each job
def updateAccessControl(job_id, group_name, permission_level):
    # Add additional permissions
    access_control = AccessControlRequest(
        group_name=group_name, 
        permission_level=permission_level
    )

    # Apply permissions to the workflow
    w.permissions.update(
        request_object_type="jobs",
        request_object_id=job_id,
        access_control_list=[access_control])

# creates a list with data for each job
def generate_dfList(jobs_permissions, job_name, job):
    rows = []
    for acl in jobs_permissions.access_control_list:
        for perm in acl.all_permissions:
            rows.append(Row(
                job_name=job_name,
                job_id=job,
                display_name=acl.display_name or acl.group_name or acl.user_name,
                service_principal_name=acl.service_principal_name,
                permission_level=perm.permission_level.name,
                inherited_from_object=perm.inherited_from_object
                ))
    return rows

In [0]:
def generateDfPermissions():
    jobs_df = spark.table(f"YOUR_CATALOG.default.workflows")

    # selects the job_id and trigger_type columns and collects the results into a list
    filt_jobs_list = jobs_df.select("job_id", "name").collect()

    # generates a table per workflow with each person’s permissions
    df_permissions_jobs = []

    for job, job_name in filt_jobs_list:

        jobs_permissions = w.jobs.get_permissions(job_id=job)
        rows = generate_dfList(jobs_permissions, job_name, job)
        df_permissions_jobs.extend(rows)

    df = spark.createDataFrame(df_permissions_jobs)

    df.write.mode("overwrite").saveAsTable(f"YOUR_CATALOG.default.workflows_permissions")

    return df

def generateListManage(df, lista_exclude, group_name):
    # for manage permission
    df_filter = (df.filter((~df.job_name.isin(lista_exclude))))

    # groups by job_id and collect all display_names in a list
    grouped_df = df_filter.groupBy("job_id").agg(collect_set("display_name").alias("display_names"))

    # filters job_ids that do not have the group in the list of display_names
    df_notHavePermission = (grouped_df.filter(~array_contains(col("display_names"), group_name))) 

    # collects the jobids that will be added to the group
    collect_jobid = df_notHavePermission.select("job_id").collect()

    # job id list
    result_list = [(row.job_id) for row in collect_jobid]

    return result_list

def generateListView(df, group_name):
    # groups by job_id and collect all display_names in a list
    grouped_df = df.groupBy("job_id").agg(collect_set("display_name").alias("display_names"))

    # filters job_ids that do not have the group in the list of display_names
    df_notHavePermission = (grouped_df.filter(~array_contains(col("display_names"), group_name))) #alterar o grupo para var

    # collects the jobids that will be added to the group
    collect_jobid = df_notHavePermission.select("job_id").collect()

    # job id list
    result_list = [(row.job_id) for row in collect_jobid]

    return result_list

In [0]:
group_canView = "GROUP_VIEW_WORKFLOWS" #defines everyone in the area to be 'CAN_VIEW'
group_canManage = "workflows_dev" # group used for those who will develop workflows
JobsNotCanManage = dbutils.widgets.get("ListJobsExclude")

# performs replace in the list passed by the workflow so that it does not disable workflows that should not receive manage permission
lista_exclude = JobsNotCanManage.split(",")

if ambiente == 'prd':
    permission_level = PermissionLevel.CAN_MANAGE_RUN
elif ambiente == 'qa':
    permission_level = PermissionLevel.CAN_MANAGE_RUN
    _lista_exclude = []
    for wkl in lista_exclude:
        var = f"[QA deploy_devops_qa] {wkl}"
        _lista_exclude.append(var)
    lista_exclude = _lista_exclude
else:
    permission_level = PermissionLevel.CAN_MANAGE

In [0]:
print("Ambiente:", ambiente)

df_wklPermissions = generateDfPermissions()

lista_canView = generateListView(df_wklPermissions, group_canView)
lista_canManage = generateListManage(df_wklPermissions, lista_exclude, group_canManage)

#### Permissão de `CAN_VIEW`

In [0]:
if lista_canView:
    for jobId in lista_canView:
        permissao = PermissionLevel.CAN_VIEW
        print(f"{permissao} permission to:", jobId)

        updateAccessControl(jobId, group_canView, permissao)
else:
    print("No workflows to add permission!")

#### Permissão de `CAN_MANAGE` em dev e `CAN_MANAGE_RUN` em PRD

In [0]:
if lista_canManage:
    for jobId in lista_canManage:
        print(f"{permissao} permission to:", jobId)

        updateAccessControl(jobId, group_canManage, permission_level)
else:
    print("No workflows to add permission!")